# 40 — Text Explainability

Generates attribution visualisations for both fine-tuned text models
(**BERT** and **DistilBERT**) using two complementary methods:

| Method | What it shows |
|---|---|
| **Layer Integrated Gradients** (Captum) | Token importance via gradient integration from a PAD-token baseline |
| **Token Occlusion** | Importance measured by replacing each token with \[MASK\] and monitoring the predicted-class probability drop |

**Depends on:** `01_preprocess_text.ipynb` (run automatically below).

**Reads checkpoints from:** `../experiments/text_unimodal/v1/<model_slug>/model/`

**Writes plots to:** `../experiments/explainability/text/<model_slug>/`

---
*Requires:* `captum>=0.7.0` — install with `pip install captum`


In [1]:
%run 01_preprocess_text.ipynb
# Imports: train_df, dev_df, test_df, label2id, id2label, labels_sorted

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Label map: {'anger': 0, 'disgust': 1, 'fear': 2, 'happiness': 3, 'neutral': 4, 'sadness': 5, 'surprise': 6}
Train: 9,989 | Dev: 1,109 | Test: 2,610
Dropped from dev (unseen labels): 0
Dropped from test (unseen labels): 0
[train] OK — 9,989 rows, dtypes: {'text': <StringDtype(na_value=nan)>, 'emotion': <StringDtype(na_value=nan)>, 'label': dtype('int64')}
[dev] OK — 1,109 rows, dtypes: {'text': <StringDtype(na_value=nan)>, 'emotion': <StringDtype(na_value=nan)>, 'label': dtype('int64')}
[test] OK — 2,610 rows, dtypes: {'text': <StringDtype(na_value=nan)>, 'emotion': <StringDtype(na_value=nan)>, 'label': dtype('int64')}


In [3]:
import os, sys, json
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, "src")) else os.path.abspath(os.path.join(_here, ".."))
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.config import EXP_ROOT, VERSION, MODELS_TO_RUN, DEVICE, EXP_ROOT_EXPLAIN
from src.training.utils import slugify
from src.explainability import TextExplainer
from src.explainability.utils import (
    make_explain_dir, select_explanation_samples,
    plot_token_attribution,
)

print("Device:", DEVICE)
print("Models:", MODELS_TO_RUN)

Device: cuda
Models: ['bert-base-uncased', 'distilbert-base-uncased']


## Load checkpoints

We load both BERT and DistilBERT from their best-dev-F1 checkpoints.


In [4]:
def load_text_model(model_name):
    slug      = slugify(model_name)
    model_dir = os.path.join(EXP_ROOT, VERSION, slug, "model")
    assert os.path.isdir(model_dir), f"Checkpoint not found: {model_dir}"
    model     = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE).eval()
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    print(f"  Loaded {model_name}  ({sum(p.numel() for p in model.parameters()):,} params)")
    return model, tokenizer

explainers = {}
for mname in MODELS_TO_RUN:
    print(f"Loading {mname} …")
    model, tokenizer = load_text_model(mname)
    explainers[mname] = TextExplainer(model, tokenizer, DEVICE)
print("All models loaded.")

Loading bert-base-uncased …


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 868.40it/s, Materializing param=classifier.weight]                                      


  Loaded bert-base-uncased  (109,487,623 params)
Loading distilbert-base-uncased …


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 987.99it/s, Materializing param=pre_classifier.weight]                                  


  Loaded distilbert-base-uncased  (66,958,855 params)
All models loaded.


## Collect test-set predictions

Run inference over the full test split for each model, then select
representative samples: **one correctly classified** and **one
misclassified** per emotion class.


In [5]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from src.data.text_preprocessing import TextEmotionDataset

@torch.no_grad()
def get_test_preds(explainer, test_df):
    tokenizer = explainer.tokenizer
    collator  = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")
    loader    = DataLoader(
        TextEmotionDataset(test_df, tokenizer),
        batch_size=32, shuffle=False, collate_fn=collator,
    )
    all_preds, all_labels = [], []
    explainer.model.eval()
    for batch in loader:
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = explainer.model(**batch).logits
        preds  = logits.argmax(dim=-1).cpu().tolist()
        lbls   = batch["labels"].cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(lbls)
    return all_preds, all_labels

model_samples = {}
for mname, explainer in explainers.items():
    print(f"Running inference: {mname} …")
    preds, labels = get_test_preds(explainer, test_df)
    acc = sum(p == l for p, l in zip(preds, labels)) / len(labels)
    print(f"  Accuracy: {acc:.4f}")
    samples = select_explanation_samples(labels, preds, id2label, n_correct=1, n_incorrect=1)
    model_samples[mname] = {"preds": preds, "labels": labels, "samples": samples}
    print(f"  Selected {sum(len(v) for v in samples.values())} samples")

Running inference: bert-base-uncased …
  Accuracy: 0.6165
  Selected 14 samples
Running inference: distilbert-base-uncased …
  Accuracy: 0.5877
  Selected 14 samples


## Generate explanations

For each model and each selected sample we produce:
- An **IG attribution** plot (Layer Integrated Gradients, 50 steps)
- A **Token Occlusion** plot (one [MASK] substitution per token)

Figures are saved to `experiments/explainability/text/<model_slug>/`.


In [6]:
IG_STEPS = 50

for mname, explainer in explainers.items():
    slug    = slugify(mname)
    out_dir = make_explain_dir("text", slug)
    samples = model_samples[mname]["samples"]
    labels  = model_samples[mname]["labels"]
    preds   = model_samples[mname]["preds"]
    texts   = test_df["text"].tolist()

    print(f"\n=== {mname} → {out_dir} ===")
    for emotion, sample_list in samples.items():
        for idx, is_correct in sample_list:
            text         = texts[idx]
            true_label   = id2label[labels[idx]]
            pred_label   = id2label[preds[idx]]
            correctness  = "correct" if is_correct else "wrong"
            sample_tag   = f"{emotion.lower()}_{correctness}_idx{idx}"

            print(f"  [{emotion}] {'✓' if is_correct else '✗'}  idx={idx}")
            print(f"    Text: {text[:80]} …" if len(text) > 80 else f"    Text: {text}")

            # ── Integrated Gradients ────────────────────────────────────────
            tokens, ig_scores, pred_cls = explainer.integrated_gradients(
                text, n_steps=IG_STEPS
            )
            title_ig = (
                f"IG  |  {mname.split('-')[0].upper()}  |  "
                f"True: {true_label}  Pred: {pred_label}"
            )
            plot_token_attribution(
                tokens, ig_scores, title_ig,
                save_path=os.path.join(out_dir, f"ig_{sample_tag}.png"),
                method_label="IG attribution (summed over embedding dim)",
            )

            # ── Token Occlusion ─────────────────────────────────────────────
            tokens_occ, occ_scores, _ = explainer.token_occlusion(
                text, target_class=pred_cls
            )
            title_occ = (
                f"Occlusion  |  {mname.split('-')[0].upper()}  |  "
                f"True: {true_label}  Pred: {pred_label}"
            )
            plot_token_attribution(
                tokens_occ, occ_scores, title_occ,
                save_path=os.path.join(out_dir, f"occ_{sample_tag}.png"),
                method_label="Occlusion score  (prob drop when masked)",
            )

    print(f"  Saved to {out_dir}")

print("\nDone — all text explanations saved.")


=== bert-base-uncased → ../experiments/explainability\text\bert_base_uncased ===
  [anger] ✓  idx=96
    Text: I mean, enough of the silliness!
  [anger] ✗  idx=1869
    Text: No-no, she kicked our butts. You could be on the Olympic standing-there team.
  [disgust] ✓  idx=2428
    Text: You are both idiots.
  [disgust] ✗  idx=972
    Text: Hey, y'know what you should do? You should take her back to the 1890's, when tha …
  [fear] ✓  idx=1724
    Text: The charity's on fire!
  [fear] ✗  idx=1335
    Text: Are you gonna break up with me if I get fat again?
  [happiness] ✓  idx=400
    Text: Oh thats so exciting!
  [happiness] ✗  idx=506
    Text: Sweetie thats it.
  [neutral] ✓  idx=1762
    Text: All right, well finish your coffee; lets go.
  [neutral] ✗  idx=1083
    Text: Okay, whatever, I really havent thought about it that much.
  [sadness] ✓  idx=1064
    Text: Well, it just hurts so bad. I uh
  [sadness] ✗  idx=772
    Text: Can we please turn this off?
  [surprise] ✓  idx=50

## Quick sanity check

Display the first IG plot inline to verify the output looks correct.


In [1]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

for mname in MODELS_TO_RUN:
    slug    = slugify(mname)
    out_dir = os.path.join(EXP_ROOT_EXPLAIN, "text", slug)
    pngs    = sorted([f for f in os.listdir(out_dir) if f.startswith("ig_")])
    if pngs:
        img = mpimg.imread(os.path.join(out_dir, pngs[0]))
        plt.figure(figsize=(8, max(3, img.shape[0] / 80)))
        plt.imshow(img)
        plt.axis("off")
        plt.title(pngs[0])
        plt.tight_layout()
        plt.show()
        break

NameError: name 'MODELS_TO_RUN' is not defined